# GLM-OCR — LoRA fine-tuning for invoice information extraction

Fine-tunes [`zai-org/GLM-OCR`](https://huggingface.co/zai-org/GLM-OCR) (1.1 B parameters) with LoRA to extract five
fields from invoice images as JSON: `CLIENT`, `FOURNISSEUR`, `DATE_FACTURE`, `MONTANT_TTC`, `NUMERO_FACTURE`.

| Setting | Value |
|---|---|
| Trainable parameters | 2,727,936 (0.25 %) |
| LoRA | r = 4, α = 8, dropout 0.2, q/k/v/o + gate/up/down projections |
| Optimiser | AdamW (fused), FP16, cosine schedule, weight decay 0.05, NEFTune α = 5 |
| Batch | 1 × 8 gradient accumulation (effective 8) |
| Images / sequence | max 1024 px, max 2048 tokens |
| Data | 3,203 training / 801 test invoices |

Training ran for about 40 epochs over several GPU sessions of at most 12 h, each resuming from the previous
checkpoint. This notebook is the **last resumed session**; the learning rate was lowered from 5e-6 to 2e-6 during
training. The full history is in [`TRAINING_LOG.md`](TRAINING_LOG.md).

> **Environment.** Kaggle notebook, NVIDIA Tesla T4 (16 GB), Python 3.12. GLM-OCR requires the development
> version of `transformers` (installed in the first cell). After the first cell, **restart the kernel**.
>
> **Data.** The datasets are private (client documents) and are not distributed. Paths of the form
> `/kaggle/input/datasets/<KAGGLE_USER>/...` refer to private Kaggle datasets and must be adapted.

> **Anonymisation.** Outputs containing client or supplier data and document images were removed.
> Only aggregate metrics are kept.

> **Language.** Comments and messages were translated into English after the run. The extraction prompt is kept in French, exactly as used for training.


## 1. Environment setup
Installs the development version of `transformers` and pinned versions of `peft`, `trl`, `accelerate`, then patches `peft` (bitsandbytes dispatch disabled) and `torchvision` (CUDA op checks) to avoid binary incompatibilities on Kaggle. Restart the kernel afterwards.

In [ ]:
import os, sys, subprocess, shutil, site
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"]        = "1"

def pip(cmd):
    r = subprocess.run([sys.executable, "-m", "pip"] + cmd,
                       capture_output=True, text=True)
    if r.returncode != 0:
        print("ERR:", r.stderr[-300:])
    return r

pip(["uninstall", "-y", "transformers", "tokenizers",
     "huggingface-hub", "bitsandbytes", "peft", "trl", "accelerate"])

r = pip(["install", "-q", "--no-cache-dir",
         "git+https://github.com/huggingface/transformers.git"])
print(("OK" if r.returncode==0 else "ERR"), "transformers @ git")

for p in ["huggingface-hub==0.28.1", "accelerate==1.3.0",
          "peft==0.13.2", "trl==0.11.4"]:
    r = pip(["install", "-q", "--no-cache-dir", p])
    print(("OK" if r.returncode==0 else "ERR"), p)

pip(["uninstall", "-y", "bitsandbytes"])
print("✅ bitsandbytes removed")

for sp in site.getsitepackages():
    bnb_file = Path(sp) / "peft" / "tuners" / "lora" / "bnb.py"
    if bnb_file.exists():
        bnb_file.write_text(
            "# patched\n"
            "def dispatch_bnb_8bit(*a, **k): return None\n"
            "def dispatch_bnb_4bit(*a, **k): return None\n"
        )
        print("✅ peft/bnb.py patched")
        break

for sp in site.getsitepackages():
    tv_dir = Path(sp) / "torchvision"
    if not tv_dir.exists(): continue
    (tv_dir / "extension.py").write_text(
        "_HAS_OPS = False\n"
        "def _has_ops(): return False\n"
        "def _check_cuda_version(): pass\n"
        "def _load_library(n): return None\n"
        "def _assert_has_ops(): return None\n"
    )
    (tv_dir / "_meta_registrations.py").write_text("# patched\n")
    (tv_dir / "__init__.py").write_text(
        '__version__ = "0.0.0-patched"\n'
        "from .extension import _HAS_OPS\n"
        "try:\n    from . import transforms\nexcept: transforms = None\n"
        "try:\n    from . import io\nexcept: io = None\n"
    )
    print("✅ torchvision patched")
    break

print("\n⚠️  RESTART THE KERNEL now!")

OK transformers @ git
OK huggingface-hub==0.28.1
OK accelerate==1.3.0
OK peft==0.13.2
OK trl==0.11.4
✅ bitsandbytes removed
✅ peft/bnb.py patched
✅ torchvision patched

⚠️  RESTART THE KERNEL now!


## 2. Version check
Prints library versions (for reproducibility) and loads the GLM-OCR processor and config without downloading the weights.

In [ ]:
import warnings, logging, torch, peft, torchvision, accelerate, transformers
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('peft').setLevel(logging.ERROR)

print(f"torch        : {torch.__version__}")
print(f"transformers : {transformers.__version__}")
print(f"peft         : {peft.__version__}")
print(f"accelerate   : {accelerate.__version__}")

from transformers import AutoProcessor, AutoConfig
p = AutoProcessor.from_pretrained("zai-org/GLM-OCR", trust_remote_code=True)
print(f"✅ AutoProcessor : {type(p).__name__}")
c = AutoConfig.from_pretrained("zai-org/GLM-OCR", trust_remote_code=True)
print(f"✅ model_type    : {c.model_type}")
del p, c
print("✅ All good")

torch        : 2.10.0+cu128
transformers : 5.6.0.dev0
peft         : 0.13.2
accelerate   : 1.3.0
✅ AutoProcessor : Glm46VProcessor
✅ model_type    : glm_ocr
✅ All good


## 3. Configuration and dataset
Hyper-parameters, dataset paths and path repair (image paths in the annotation files are re-indexed by file name).

In [ ]:
import os, json, subprocess, torch
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"]        = "1"

KAGGLE_USERNAME = '<KAGGLE_USER>'
SESSION_ID      = 'v9s9'

BASE_DATA  = Path('/kaggle/input/datasets/<KAGGLE_USER>/glmdata')
IMAGES_DIR = BASE_DATA / 'images'
TRAIN_JSON = BASE_DATA / 'output' / 'train_glmocr.json'
TEST_JSON  = BASE_DATA / 'output' / 'test_glmocr.json'
WORK_DIR   = Path('/kaggle/working')
OUTPUT_DIR = WORK_DIR / 'glmocr_finetuned'
BEST_DIR   = WORK_DIR / 'glmocr_best'
OUTPUT_DIR.mkdir(exist_ok=True)
BEST_DIR.mkdir(exist_ok=True)

MODEL_NAME              = 'zai-org/GLM-OCR'
NUM_EPOCHS              = 50
EARLY_STOPPING_PATIENCE = 25
BATCH_SIZE              = 1
GRAD_ACCUM              = 8
LEARNING_RATE           = 2e-6
LORA_RANK               = 4
LORA_ALPHA              = 8
SEED                    = 42
MAX_SIZE                = 1024

gpu_ok = torch.cuda.is_available()
print(f"GPU : {torch.cuda.get_device_name(0) if gpu_ok else 'CPU only'}")

IMAGE_INDEX = {}
for p in IMAGES_DIR.rglob("*"):
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]:
        IMAGE_INDEX[p.name] = p
print(f"Indexed images: {len(IMAGE_INDEX)}")

def fix_paths(data, name):
    fixed, ko = [], 0
    for item in data:
        img = item["images"][0]
        if Path(img).exists():
            fixed.append(item); continue
        img_name = img.replace("\\", "/").split("/")[-1]
        if img_name in IMAGE_INDEX:
            it = dict(item)
            it["images"] = [str(IMAGE_INDEX[img_name])]
            fixed.append(it)
        else:
            ko += 1
    print(f"[{name}] OK={len(fixed)} KO={ko}")
    return fixed

train_data  = json.loads(Path(TRAIN_JSON).read_text())
test_data   = json.loads(Path(TEST_JSON).read_text())
train_fixed = fix_paths(train_data, "TRAIN")
test_fixed  = fix_paths(test_data,  "TEST")

DATASET_WORK = WORK_DIR / "dataset"
DATASET_WORK.mkdir(exist_ok=True)
(DATASET_WORK/"train_glmocr.json").write_text(
    json.dumps(train_fixed, ensure_ascii=False), encoding="utf-8")
(DATASET_WORK/"test_glmocr.json").write_text(
    json.dumps(test_fixed, ensure_ascii=False), encoding="utf-8")

print(f"Train={len(train_fixed)} Test={len(test_fixed)}")
r = subprocess.run(["df","-h","/kaggle/working"], capture_output=True, text=True)
print(f"💾 Disk:\n{r.stdout}")
print(f'\n📋 Config {SESSION_ID} :')
print(f'   MAX_SIZE={MAX_SIZE}px LR={LEARNING_RATE}')
print(f'   rank={LORA_RANK} NEFTune=5 weight_decay=0.05')
print(f'   NUM_EPOCHS={NUM_EPOCHS} PATIENCE={EARLY_STOPPING_PATIENCE}')

GPU : Tesla T4
Indexed images: 4004
[TRAIN] OK=3203 KO=0
[TEST] OK=801 KO=0
Train=3203 Test=801
💾 Disk:
Filesystem      Size  Used Avail Use% Mounted on
/dev/loop1       20G  2.7M   20G   1% /kaggle/working


📋 Config v9s9 :
   MAX_SIZE=1024px LR=2e-06
   rank=4 NEFTune=5 weight_decay=0.05
   NUM_EPOCHS=50 PATIENCE=25


## 4. Checkpointing callback
Saves the best adapter (lowest evaluation loss) and a resumable checkpoint after every epoch, so that training can continue across GPU sessions limited to 12 h.

In [ ]:
import time, json, shutil, subprocess
from pathlib import Path
import torch
from transformers import TrainerCallback

def upload_to_kaggle(folder_path, dataset_name):
    folder = Path(folder_path)
    folder.mkdir(parents=True, exist_ok=True)
    metadata = {
        "title":    dataset_name,
        "id":       f"{KAGGLE_USERNAME}/{dataset_name}",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(folder / 'dataset-metadata.json', 'w') as f:
        json.dump(metadata, f)
    r = subprocess.run(
        ['kaggle', 'datasets', 'create',
         '-p', str(folder), '--dir-mode', 'zip'],
        capture_output=True, text=True, timeout=300
    )
    if r.returncode == 0:
        print(f'   ☁️  Created: {dataset_name}'); return
    r2 = subprocess.run(
        ['kaggle', 'datasets', 'version',
         '-p', str(folder), '-m', 'update', '--dir-mode', 'zip'],
        capture_output=True, text=True, timeout=300
    )
    print(f'   {"☁️  Updated" if r2.returncode==0 else "⚠️  Failed"}: {dataset_name}')

class KaggleSaver(TrainerCallback):
    def __init__(self, output_dir, best_dir, patience=25,
                 max_hours=11.0, resume_every_epochs=1):
        self.output_dir          = Path(output_dir)
        self.best_dir            = Path(best_dir)
        self.processor           = None
        self.patience            = patience
        self.no_improve          = 0
        self.best_score          = float("inf")
        self.best_epoch          = 0
        self.start_time          = time.time()
        self.max_seconds         = max_hours * 3600
        self.resume_every_epochs = resume_every_epochs
        self.last_resume_epoch   = -1
        self.emergency_done      = False

    def _latest_checkpoint(self):
        cps = sorted(self.output_dir.glob("checkpoint-*"))
        return cps[-1] if cps else None

    def zip_best(self, epoch, model):
        self.best_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(str(self.best_dir))
        if self.processor:
            self.processor.save_pretrained(str(self.best_dir))
        for old in Path("/kaggle/working").glob("glmocr_best_*.zip"):
            old.unlink()
        name = f"glmocr-{SESSION_ID}-best-ep{epoch:02d}"
        shutil.make_archive(f"/kaggle/working/{name}", "zip", str(self.best_dir))
        sz = Path(f"/kaggle/working/{name}.zip").stat().st_size/1e6
        print(f"📦 {name}.zip ({sz:.0f} MB) [BEST]")
        upload_to_kaggle(str(self.best_dir), name)

    def zip_resume(self, epoch):
        cp = self._latest_checkpoint()
        if cp is None:
            print("⚠️ No checkpoint."); return
        (cp/"callback_state.json").write_text(json.dumps({
            "best_score": self.best_score,
            "best_epoch": self.best_epoch,
            "no_improve": self.no_improve,
            "epoch":      epoch,
            "session_id": SESSION_ID,
        }, indent=2))
        for old in Path("/kaggle/working").glob("glmocr_resume_*.zip"):
            old.unlink()
        name     = f"glmocr-{SESSION_ID}-resume-ep{epoch:02d}"
        zip_path = f"/kaggle/working/{name}.zip"
        shutil.make_archive(f"/kaggle/working/{name}", "zip", str(cp))
        sz = Path(zip_path).stat().st_size/1e6
        print(f"📦 {name}.zip ({sz:.0f} MB) [RESUME]")
        resume_dir = Path(f"/kaggle/working/resume_ep{epoch:02d}")
        if resume_dir.exists(): shutil.rmtree(resume_dir)
        resume_dir.mkdir()
        shutil.copy(zip_path, resume_dir / f"{name}.zip")
        upload_to_kaggle(str(resume_dir), name)

    def on_evaluate(self, args, state, control,
                    metrics=None, model=None, **kwargs):
        if not metrics or model is None: return
        epoch     = int(state.epoch or 0)
        eval_loss = metrics.get("eval_loss")
        if eval_loss is not None:
            if eval_loss < self.best_score:
                self.best_score = eval_loss
                self.best_epoch = epoch
                self.no_improve = 0
                print(f"✅ BEST eval_loss={eval_loss:.4f} epoch={epoch}")
                self.zip_best(epoch, model)
            else:
                self.no_improve += 1
                print(f"⏳ ({self.no_improve}/{self.patience}) "
                      f"eval_loss={eval_loss:.4f} best={self.best_score:.4f}")
        elapsed   = time.time() - self.start_time
        remaining = self.max_seconds - elapsed
        if remaining < 3600 and not self.emergency_done:
            print(f"🚨 Emergency save ({remaining/60:.0f} min restantes)")
            self.zip_resume(epoch)
            self.emergency_done = True
        if self.no_improve >= self.patience:
            print("🛑 Early stopping.")
            self.zip_resume(epoch)
            control.should_training_stop = True
        torch.cuda.empty_cache()

    def on_save(self, args, state, control, **kwargs):
        epoch = int(state.epoch or 0)
        if (epoch > 0 and epoch % self.resume_every_epochs == 0
                and self.last_resume_epoch != epoch):
            self.zip_resume(epoch)
            self.last_resume_epoch = epoch
        for cp in sorted(self.output_dir.glob("checkpoint-*"))[:-1]:
            shutil.rmtree(cp, ignore_errors=True)
        return control

disk_saver = KaggleSaver(
    OUTPUT_DIR, BEST_DIR,
    patience            = EARLY_STOPPING_PATIENCE,
    max_hours           = 11.0,
    resume_every_epochs = 1
)
print("✅ KaggleSaver ready.")

✅ KaggleSaver ready.


## 5. Model, LoRA adapters and dataset
Loads GLM-OCR in FP16, attaches LoRA adapters (r = 4, α = 8, dropout 0.2 on all attention and MLP projections) and builds the dataset from a pre-tokenised cache.

In [ ]:
import warnings, logging, torch, json
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('peft').setLevel(logging.ERROR)

import peft
bnb_file = Path(peft.__file__).parent / 'tuners' / 'lora' / 'bnb.py'
if bnb_file.exists():
    bnb_file.write_text(
        '# disabled\n'
        'def dispatch_bnb_8bit(*a, **k): return None\n'
        'def dispatch_bnb_4bit(*a, **k): return None\n'
    )
import peft.tuners.lora.model as lora_model
lora_model.is_bnb_available      = lambda: False
lora_model.is_bnb_4bit_available = lambda: False
print('✅ bitsandbytes disabled')

from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoProcessor, AutoModelForImageTextToText
torch.cuda.empty_cache()

print(f'⏳ Loading {MODEL_NAME}...')
processor = AutoProcessor.from_pretrained(
    MODEL_NAME, use_fast=False, trust_remote_code=True)
print(f'✅ Processor : {type(processor).__name__}')
disk_saver.processor = processor

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if gpu_ok else torch.float32,
    device_map='auto' if gpu_ok else None,
    trust_remote_code=True,
)
model.config.use_cache = False
print(f'✅ Model: {sum(p.numel() for p in model.parameters()):,} params')

lora_cfg = LoraConfig(
    r=LORA_RANK, lora_alpha=LORA_ALPHA,
    lora_dropout=0.2,
    bias='none', task_type=TaskType.CAUSAL_LM,
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora_cfg)

try:
    model.enable_input_require_grads()
    model.base_model.model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    print('✅ Gradient checkpointing enabled')
except Exception as e:
    print(f'⚠️  Gradient checkpointing skipped: {e}')

model.print_trainable_parameters()

PROMPT = (
    "Extrais les informations suivantes de cette facture en JSON :\n"
    "- CLIENT : nom du client\n"
    "- FOURNISSEUR : nom du fournisseur\n"
    "- DATE_FACTURE : date de la facture (format JJ/MM/AAAA)\n"
    "- MONTANT_TTC : montant total TTC (ou N/A si absent)\n"
    "- NUMERO_FACTURE : numero de facture\n"
    "Reponds UNIQUEMENT avec un objet JSON valide.\n"
    "Si une information est absente, mettre N/A."
)

MAX_SEQ_LEN = 2048

# ── Automatic cache detection ─────────────────────────────
CACHE_INPUT = Path('/kaggle/input/datasets/<KAGGLE_USER>/glmocr-tokens-cache-1024-v2s1')
CACHE_WORK  = Path('/kaggle/working/tokens_cache')

if CACHE_INPUT.exists() and (CACHE_INPUT/'train').exists() \
        and list((CACHE_INPUT/'train').glob('*.pt')):
    CACHE_DIR = CACHE_INPUT
    print(f'✅ Cache from INPUT: {CACHE_DIR}')
elif CACHE_WORK.exists() and (CACHE_WORK/'train').exists() \
        and list((CACHE_WORK/'train').glob('*.pt')):
    CACHE_DIR = CACHE_WORK
    print(f'✅ Cache from WORKING: {CACHE_DIR}')
else:
    raise FileNotFoundError('❌ Cache introuvable !')

class CachedDataset(Dataset):
    def __init__(self, cache_dir, split, processor):
        self.files     = sorted((Path(cache_dir)/split).glob('*.pt'))
        self.processor = processor
        print(f'✅ {split} : {len(self.files)} fichiers')

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        sample        = torch.load(self.files[idx], weights_only=False)
        orig_img_path = sample.pop('orig_image_path', None)
        if orig_img_path:
            try:
                image = Image.open(orig_img_path).convert('RGB')
                w, h  = image.size
                if max(w, h) > MAX_SIZE:
                    ratio = MAX_SIZE / max(w, h)
                    image = image.resize(
                        (int(w*ratio), int(h*ratio)), Image.LANCZOS)
                msgs = [{"role": "user", "content": [
                    {"type": "image", "image": image},
                    {"type": "text",  "text":  PROMPT},
                ]}]
                pv = self.processor.apply_chat_template(
                    msgs, tokenize=True, add_generation_prompt=True,
                    return_dict=True, return_tensors='pt',
                    processor_kwargs={
                        "return_mm_token_type_ids": True,
                        "truncation": False, "padding": False,
                    },
                )
                if 'pixel_values' in pv:
                    sample['pixel_values']   = pv['pixel_values'].squeeze(0)
                if 'image_grid_thw' in pv:
                    sample['image_grid_thw'] = pv['image_grid_thw'].squeeze(0)
            except:
                pass
        return sample

def collate_fn(batch):
    b   = batch[0]
    out = {}
    for k, v in b.items():
        if isinstance(v, torch.Tensor):
            if k in ['input_ids','labels','attention_mask','mm_token_type_ids']:
                if v.shape[0] > MAX_SEQ_LEN:
                    v = v[:MAX_SEQ_LEN]
            out[k] = v.unsqueeze(0)
    return out

train_dataset = CachedDataset(CACHE_DIR, 'train', processor)
test_dataset  = CachedDataset(CACHE_DIR, 'test',  processor)

print('🔍 Test DataLoader...')
batch = next(iter(DataLoader(
    train_dataset, batch_size=1, num_workers=0, collate_fn=collate_fn
)))
for k, v in batch.items():
    print(f'   {k:<25} : {v.shape} {v.dtype}')
n_visible = (batch['labels'][0] != -100).sum().item()
print(f'\n   Labels visibles : {n_visible}')
print(f'   pixel_values    : {batch["pixel_values"].shape}')
print(f'   seq_len         : {batch["input_ids"].shape[1]}')
assert 'mm_token_type_ids' in batch, '❌ mm_token_type_ids manquant'
assert 'pixel_values'      in batch, '❌ pixel_values manquant'
assert n_visible > 0,                '❌ Aucun label visible'
print(f'\n✅ DataLoader OK — cache : {CACHE_DIR}')
del batch
torch.cuda.empty_cache()

✅ bitsandbytes disabled
⏳ Loading zai-org/GLM-OCR...
✅ Processor : Glm46VProcessor


Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

✅ Model: 1,107,405,824 params
✅ Gradient checkpointing enabled
trainable params: 2,727,936 || all params: 1,110,133,760 || trainable%: 0.2457
✅ Cache from INPUT: /kaggle/input/datasets/<KAGGLE_USER>/glmocr-tokens-cache-1024-v2s1
✅ train : 3203 fichiers
✅ test : 801 fichiers
🔍 Test DataLoader...
   input_ids                 : torch.Size([1, 735]) torch.int64
   attention_mask            : torch.Size([1, 735]) torch.int64
   labels                    : torch.Size([1, 735]) torch.int64
   mm_token_type_ids         : torch.Size([1, 735]) torch.int64
   image_grid_thw            : torch.Size([1, 3]) torch.int64
   pixel_values              : torch.Size([1, 2116, 1176]) torch.float32

   Labels visibles : 87
   pixel_values    : torch.Size([1, 2116, 1176])
   seq_len         : 735

✅ DataLoader OK — cache : /kaggle/input/datasets/<KAGGLE_USER>/glmocr-tokens-cache-1024-v2s1


## 6. Training (resumed session)
Resumes training from the last checkpoint of the previous session, restoring the callback state (best loss, patience counter).

In [ ]:
import os, shutil, json, warnings, logging, gc
from pathlib import Path
from transformers import TrainingArguments, Trainer, set_seed, AutoModelForImageTextToText
from peft import PeftModel
import torch

warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('peft').setLevel(logging.ERROR)

set_seed(SEED)
os.environ["WANDB_DISABLED"]         = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ALLOC_CONF"]     = "expandable_segments:True"
if gpu_ok:
    torch.backends.cuda.matmul.allow_tf32 = True

gc.collect()
torch.cuda.empty_cache()
free = torch.cuda.get_device_properties(0).total_memory/1e9 \
     - torch.cuda.memory_allocated()/1e9
print(f'🖥️  Free VRAM: {free:.1f}GB')

# ── Resume from local epoch 22 — best=0.0109 at local epoch 13 ──
BEST_CKPT   = Path('/kaggle/input/datasets/<KAGGLE_USER>/glmocr-v7s7-best-ep13')
RESUME_CKPT = Path('/kaggle/input/datasets/<KAGGLE_USER>/glmocr-v7s7-resume-ep22')

print(f'🔄 Resuming from local epoch 22 — session {SESSION_ID}...')
print(f'   best so far: 0.0109 at local epoch 13 — no_improve=10/25')

base = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if gpu_ok else torch.float32,
    device_map='auto' if gpu_ok else None,
    trust_remote_code=True,
)
base.config.use_cache = False

model = PeftModel.from_pretrained(
    base, str(RESUME_CKPT), is_trainable=True
)

try:
    model.enable_input_require_grads()
    model.base_model.model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    print('✅ Gradient checkpointing enabled')
except Exception as e:
    print(f'⚠️  Gradient checkpointing skipped: {e}')

model.print_trainable_parameters()

# ── Restaurer callback ────────────────────────────────────
state_file = RESUME_CKPT / 'callback_state.json'
if state_file.exists():
    state = json.loads(state_file.read_text())
    disk_saver.best_score = state['best_score']
    disk_saver.best_epoch = state['best_epoch']
    disk_saver.no_improve = state['no_improve']
    print(f'✅ Callback restored from callback_state.json:')
else:
    disk_saver.best_score = 0.0109
    disk_saver.best_epoch = 13
    disk_saver.no_improve = 10
    print(f'⚠️  Valeurs manuelles :')

print(f'   best_score = {disk_saver.best_score:.4f}')
print(f'   best_epoch = {disk_saver.best_epoch}')
print(f'   no_improve = {disk_saver.no_improve}/25')

disk_saver.processor      = processor
disk_saver.emergency_done = False

# ── Custom Trainer anti-OOM ───────────────────────────────
class LowMemTrainer(Trainer):
    def training_step(self, model, inputs, num_items_in_batch=None):
        torch.cuda.empty_cache()
        return super().training_step(model, inputs, num_items_in_batch)

# ── Training Arguments ────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = 1,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,  # 2e-6
    warmup_steps                = 50,
    lr_scheduler_type           = 'cosine',
    weight_decay                = 0.05,
    neftune_noise_alpha         = 5,
    fp16                        = gpu_ok,
    max_grad_norm               = 1.0,
    logging_steps               = 20,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    save_total_limit            = 1,
    load_best_model_at_end      = False,
    report_to                   = 'none',
    seed                        = SEED,
    remove_unused_columns       = False,
    dataloader_num_workers      = 0,
    dataloader_pin_memory       = False,
    gradient_checkpointing      = False,
    optim                       = 'adamw_torch_fused',
)

trainer = LowMemTrainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = test_dataset,
    data_collator = collate_fn,
    callbacks     = [disk_saver],
)

print(f'\n🚀 Fine-tuning GLM-OCR — RESUMED {SESSION_ID}')
print(f'   Train={len(train_dataset)} Test={len(test_dataset)}')
print(f'   LR={LEARNING_RATE} rank={LORA_RANK} MAX_SIZE={MAX_SIZE}px')
print(f'   NEFTune=5 weight_decay=0.05 warmup_steps=50')
print(f'   Best so far: {disk_saver.best_score:.4f} (epoch {disk_saver.best_epoch})')
print(f'   No improve : {disk_saver.no_improve}/25 — {25-disk_saver.no_improve} epochs of patience left')
print(f'   Auto upload: ✅ ({KAGGLE_USERNAME})')

try:
    train_result = trainer.train(
        resume_from_checkpoint=str(RESUME_CKPT)
    )

except torch.cuda.OutOfMemoryError:
    print('\n❌ OOM')
    trainer.model.save_pretrained(str(BEST_DIR))
    processor.save_pretrained(str(BEST_DIR))
    upload_to_kaggle(str(BEST_DIR), f'glmocr-{SESSION_ID}-oom-backup')
    raise

except Exception as e:
    print(f'\n❌ Crash : {e}')
    trainer.model.save_pretrained(str(BEST_DIR))
    processor.save_pretrained(str(BEST_DIR))
    upload_to_kaggle(str(BEST_DIR), f'glmocr-{SESSION_ID}-crash-backup')
    raise

# ── Sauvegarde finale ─────────────────────────────────────
processor.save_pretrained(str(BEST_DIR))
zip_path = f'/kaggle/working/glmocr_{SESSION_ID}_final_ep{disk_saver.best_epoch:02d}'
shutil.make_archive(zip_path, 'zip', str(BEST_DIR))
upload_to_kaggle(str(BEST_DIR),
                 f'glmocr-{SESSION_ID}-final-ep{disk_saver.best_epoch:02d}')

print(f'\n✅ Finished {SESSION_ID}!')
print(f'   Best : {disk_saver.best_score:.4f} (epoch {disk_saver.best_epoch})')
print(f'   Kaggle   : kaggle.com/datasets/{KAGGLE_USERNAME}/'
      f'glmocr-{SESSION_ID}-final-ep{disk_saver.best_epoch:02d}')

🖥️  Free VRAM: 13.9GB
🔄 Resuming from local epoch 22 — session v9s9...
   best so far: 0.0109 at local epoch 13 — no_improve=10/25


Loading weights:   0%|          | 0/510 [00:00<?, ?it/s]

✅ Gradient checkpointing enabled
trainable params: 2,727,936 || all params: 1,110,133,760 || trainable%: 0.2457
✅ Callback restored from callback_state.json:
   best_score = 0.0109
   best_epoch = 13
   no_improve = 10/25

🚀 Fine-tuning GLM-OCR — RESUMED v9s9
   Train=3203 Test=801
   LR=2e-06 rank=4 MAX_SIZE=1024px
   NEFTune=5 weight_decay=0.05 warmup_steps=50
   Best so far: 0.0109 (epoch 13)
   No improve : 10/25 — 15 epochs of patience left
   Auto upload: ✅ (<KAGGLE_USER>)
{'loss': '0.0006704', 'grad_norm': '0.002537', 'learning_rate': '3.126e-06', 'epoch': '21.05'}
{'loss': '0.0006942', 'grad_norm': '0.004506', 'learning_rate': '3.118e-06', 'epoch': '21.1'}
{'loss': '0.0005757', 'grad_norm': '0.007186', 'learning_rate': '3.111e-06', 'epoch': '21.15'}
{'loss': '0.0006242', 'grad_norm': '0.006441', 'learning_rate': '3.103e-06', 'epoch': '21.2'}
{'loss': '0.0006148', 'grad_norm': '0.004075', 'learning_rate': '3.095e-06', 'epoch': '21.25'}
{'loss': '0.000669', 'grad_norm': '0.002835